In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

# --- 1. CONFIGURATION & DATA ---
# Graph Structure: (Start Node, End Node, Capacity)
# Node 0 = Source, Node 3 = Sink
# We use the same "Diamond" graph as before
edges_data = [
    (0, 1, 10.0), # Source -> 1
    (0, 2, 10.0), # Source -> 2
    (1, 2, 5.0),  # The "bridge" (Undirected)
    (1, 3, 8.0),  # 1 -> Sink
    (2, 3, 8.0)   # 2 -> Sink
]
NUM_NODES = 4
SOURCE_NODE = 0
SINK_NODE = 3

# Hyperparameters
LEARNING_RATE = 0.1
EPOCHS = 2000
LAMBDA_PHYSICS = 100.0  # Penalty weight for breaking laws of physics



In [3]:
# --- 2. HELPER FUNCTIONS (Matrix Setup) ---
def build_incidence_matrix(edges, num_nodes):
    indices = []
    values = []
    for i, (u, v, _) in enumerate(edges):
        # Column indices for the two directions of this edge
        idx_fwd, idx_bwd = 2 * i, 2 * i + 1
        
        # u -> v (u loses, v gains)
        indices.extend([[u, idx_fwd], [v, idx_fwd]])
        values.extend([-1.0, 1.0])
        
        # v -> u (v loses, u gains)
        indices.extend([[v, idx_bwd], [u, idx_bwd]])
        values.extend([-1.0, 1.0])

    i = torch.LongTensor(indices).t()
    v = torch.FloatTensor(values)
    return torch.sparse_coo_tensor(i, v, size=(num_nodes, 2 * len(edges)))

def vectorized_flow_loss(flows, incidence_matrix, capacities, source, sink, lambda_phys):
    # 1. Calculate Net Flow for all nodes (Matrix Mult)
    # Flatten flows to (2*E, 1) for multiplication
    flat_flows = flows.view(-1).unsqueeze(1) 
    net_flows = torch.sparse.mm(incidence_matrix, flat_flows).squeeze()

    # 2. Conservation Loss (Nodes must balance to 0, except Source/Sink)
    mask = torch.ones(net_flows.shape[0], dtype=torch.bool)
    mask[source] = False
    mask[sink] = False
    loss_cons = torch.sum(net_flows[mask] ** 2)

    # 3. Capacity Loss (Sum of both dirs must be <= Capacity)
    # Sum flow in both directions: shape (E, 2) -> (E,)
    total_edge_flow = torch.sum(flows, dim=1) 
    loss_cap = torch.sum(torch.relu(total_edge_flow - capacities) ** 2)

    # 4. Objective: Maximize Flow into Sink
    # We minimize negative flow
    flow_at_sink = net_flows[sink]
    loss_obj = -flow_at_sink

    return (lambda_phys * loss_cons) + (lambda_phys * loss_cap) + loss_obj, flow_at_sink



In [10]:
edges_data

[(0, 1, 10.0), (0, 2, 10.0), (1, 2, 5.0), (1, 3, 8.0), (2, 3, 8.0)]

In [3]:
# --- 3. THE MODEL ---
class FlowNetwork(nn.Module):
    def __init__(self, num_edges):
        super().__init__()
        # Parameters: Raw numbers that can be negative
        self.raw_weights = nn.Parameter(torch.randn(num_edges, 2))
    
    def forward(self):
        # Apply ReLU so the solver physically cannot output negative flow
        return torch.relu(self.raw_weights)



In [12]:
incidence_matrix = build_incidence_matrix(edges_data, NUM_NODES)
incidence_matrix

tensor(indices=tensor([[0, 1, 1, 0, 0, 2, 2, 0, 1, 2, 2, 1, 1, 3, 3, 1, 2, 3, 3,
                        2],
                       [0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9,
                        9]]),
       values=tensor([-1.,  1., -1.,  1., -1.,  1., -1.,  1., -1.,  1., -1.,
                       1., -1.,  1., -1.,  1., -1.,  1., -1.,  1.]),
       size=(4, 10), nnz=20, layout=torch.sparse_coo)

In [4]:
# --- 4. TRAINING SETUP ---
# Pre-compute static data (Matrix & Capacities)
incidence_matrix = build_incidence_matrix(edges_data, NUM_NODES)
capacities = torch.tensor([e[2] for e in edges_data], dtype=torch.float32)

model = FlowNetwork(len(edges_data))
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [5]:
# --- 5. THE TRAINING LOOP ---
print(f"{'Epoch':<10} | {'Loss':<10} | {'Sink Flow':<10} | {'Status'}")
print("-" * 50)

for epoch in range(EPOCHS):
    optimizer.zero_grad()  # Reset gradients
    
    # Forward Pass
    current_flows = model()
    
    # Calculate Loss
    loss, sink_flow = vectorized_flow_loss(
        current_flows, incidence_matrix, capacities, 
        SOURCE_NODE, SINK_NODE, LAMBDA_PHYSICS
    )
    
    # Backward Pass (Compute Gradients)
    loss.backward()
    
    # Update Weights
    optimizer.step()
    
    # Monitoring
    if epoch % 500 == 0 or epoch == EPOCHS - 1:
        print(f"{epoch:<10} | {loss.item():.4f}     | {sink_flow.item():.4f}     | Training...")

# --- 6. INSPECT RESULTS ---
print("\n--- FINAL RESULTS ---")
final_flows = model().detach() # Detach from gradient graph for printing
total_sink_flow = 0

for i, (u, v, cap) in enumerate(edges_data):
    f_uv = final_flows[i, 0].item()
    f_vu = final_flows[i, 1].item()
    
    # Only print significant flows
    if f_uv > 0.01 or f_vu > 0.01:
        print(f"Edge {u}-{v} (Max {cap}): \t{f_uv:.2f} -> \t{f_vu:.2f} <-")

print(f"\nFinal Calculated Max Flow: {vectorized_flow_loss(final_flows, incidence_matrix, capacities, SOURCE_NODE, SINK_NODE, 0)[1].item():.2f}")

Epoch      | Loss       | Sink Flow  | Status
--------------------------------------------------
0          | 390.8421     | 0.7250     | Training...
500        | -0.0025     | 0.0055     | Training...
1000       | -0.0025     | 0.0050     | Training...
1500       | -0.0025     | 0.0050     | Training...
1999       | -0.0025     | 0.0050     | Training...

--- FINAL RESULTS ---
Edge 2-3 (Max 8.0): 	0.55 -> 	0.55 <-

Final Calculated Max Flow: 0.01
